[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch08-model-training/02-bo-toi-uu-va-batch.ipynb)

# Bộ tối ưu và batch: quỹ đạo, trạng thái, lịch tốc độ học và tích luỹ gradient

Chương 8 nhìn bộ tối ưu (optimizer) bằng con mắt hệ thống: nó dịch gradient thành bước cập nhật,
nhưng nó cũng **giữ trạng thái**, và trạng thái ấy chiếm bộ nhớ ngang hoặc hơn chính trọng số.
Kích thước batch thì vừa là một tham số thống kê vừa là một tham số hệ thống. Sổ tay này làm cả
hai thứ thành con số.

**Bạn sẽ làm:**
1. thả bốn bộ tối ưu của chương lên cùng một mặt mất mát và xem quỹ đạo của chúng;
2. đếm trạng thái mà từng bộ tối ưu của PyTorch thật sự giữ, rồi nhân lên cỡ GPT-2 XL;
3. đo nhiễu của gradient minibatch theo kích thước batch;
4. vẽ lịch tốc độ học (learning rate schedule) của GPT-2 XL và xem vì sao trạng thái của nó phải
   vào checkpoint;
5. kiểm rằng tích luỹ gradient (gradient accumulation) cho đúng gradient của batch lớn, tìm hai
   cách làm nó sai, rồi tính kế hoạch tích luỹ của chương cho GPT-2.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Bốn bộ tối ưu trên một mặt mất mát

Mặt mất mát đồ chơi $f(w_1, w_2) = \tfrac12\,(0.2\,w_1^2 + 20\,w_2^2)$ có cùng một cực tiểu ở gốc,
nhưng hai chiều có thang gradient chênh nhau 100 lần: dốc đứng theo $w_2$, gần như phẳng theo $w_1$.
Đây đúng là tình huống chương mô tả:

* **SGD** áp một tốc độ học (learning rate) cho mọi tham số. Tốc độ học đủ lớn cho chiều phẳng thì
  quá lớn cho chiều dốc.
* **Động lượng** (momentum) tích luỹ một vector vận tốc qua các vòng lặp, làm mượt hướng dao động.
* **RMSprop** giữ trung bình trượt của gradient bình phương cho mỗi tham số, và chia bước cho căn
  của nó.
* **Adam** làm cả hai việc, với hệ số suy giảm $\beta_1$ và $\beta_2$.

Ô dưới dùng chính các lớp `torch.optim` mà bạn dùng để huấn luyện mạng thật. Tốc độ học của mỗi
bộ được chọn bằng tay cho mặt này, nên đây là minh hoạ hình dạng quỹ đạo, không phải cuộc thi.

**Dự đoán:** sau 40 bước, bộ nào vẫn còn xa cực tiểu theo chiều $w_1$? Bộ nào dập dao động theo
$w_2$ nhanh nhất?

In [ ]:
BETA1 = sach(0.9, nguon="β₁=0,9 · β₂=0,999")
BETA2 = sach(0.999, nguon="β₁=0,9 · β₂=0,999")


def f_do_choi(w):
    return 0.5 * (0.2 * w[0] ** 2 + 20 * w[1] ** 2)


def quy_dao(tao_bo_toi_uu, buoc=40):
    w = torch.tensor([-4.5, 1.2], requires_grad=True)
    bo = tao_bo_toi_uu([w])
    duong = [w.detach().clone()]
    for _ in range(buoc):
        bo.zero_grad()
        f_do_choi(w).backward()
        bo.step()
        duong.append(w.detach().clone())
    return torch.stack(duong).numpy()


bon_bo = {
    "SGD, lr 0.09": lambda p: torch.optim.SGD(p, lr=0.09),
    "động lượng 0.9, lr 0.03": lambda p: torch.optim.SGD(p, lr=0.03, momentum=0.9),
    "RMSprop, lr 0.05": lambda p: torch.optim.RMSprop(p, lr=0.05),
    "Adam, lr 0.2": lambda p: torch.optim.Adam(p, lr=0.2, betas=(BETA1, BETA2)),
}
duong = {ten: quy_dao(tao) for ten, tao in bon_bo.items()}
for ten, d in duong.items():
    print(f"{ten:<26} sau 40 bước: w = ({d[-1, 0]:+.3f}, {d[-1, 1]:+.3f}), f = {f_do_choi(d[-1]):.4f}")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
g1, g2 = np.meshgrid(np.linspace(-5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
ax.contour(g1, g2, 0.5 * (0.2 * g1**2 + 20 * g2**2), levels=np.geomspace(0.05, 25, 12), colors="0.8", linewidths=0.7)
for (ten, d), mau in zip(duong.items(), ("C0", "C1", "C2", "C3")):
    ax.plot(d[:, 0], d[:, 1], ".-", color=mau, lw=1, ms=3, label=ten)
    ax2.semilogy(range(len(d)), [f_do_choi(w) for w in d], color=mau, label=ten)
ax.plot([-4.5], [1.2], "ks", ms=5)
ax.annotate("điểm xuất phát", (-4.5, 1.2), xytext=(8, 4), textcoords="offset points", fontsize=8)
ax.plot([0], [0], "k*", ms=10)
ax.set_xlabel("w₁ (chiều phẳng)")
ax.set_ylabel("w₂ (chiều dốc)")
ax.set_title("Quỹ đạo trên f = ½(0.2 w₁² + 20 w₂²)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8, frameon=False)
ax2.set_xlabel("bước")
ax2.set_ylabel("f (thang log)")
ax2.set_title("Giá trị mất mát")
plt.tight_layout()
plt.show()

SGD dao động theo $w_2$ và bò theo $w_1$: một tốc độ học không vừa cả hai thang. Động lượng tích
đà theo chiều phẳng nên tới gần cực tiểu nhanh hơn, nhưng đà ấy cũng làm nó vượt qua rồi quay lại.
RMSprop chia mỗi chiều cho thang gradient của chính chiều đó, nên dao động theo $w_2$ tắt gần như
ngay, và hai chiều đi với cùng một nhịp. Adam ghép đà với việc chia thang ấy.

Chương nhấn mạnh một điều mà hình này không chứng minh được: lợi ích hội tụ của Adam so với SGD hay
động lượng phải đo trên cùng mô hình, dữ liệu, lịch, batch và tiêu chí dừng. Còn **cái giá** của nó
thì đếm được ngay, không cần huấn luyện gì.

## 2. Trạng thái mà bộ tối ưu giữ

Mỗi bộ tối ưu của PyTorch giữ trạng thái bộ tối ưu (optimizer state) trong `optimizer.state`, một
từ điển theo từng tham số. Ô dưới chạy **một** bước của từng bộ trên cùng một mạng 784-256-256-10
rồi đếm byte của mọi tensor trạng thái có cùng hình dạng với tham số của nó.

**Dự đoán:** với Adam, tổng tham số cộng trạng thái là mấy lần dung lượng tham số?

In [ ]:
nn = torch.nn
ham_mat_mat = nn.CrossEntropyLoss()


def mlp_784():
    dat_hat_giong()
    return nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))


x = torch.randn(32, 784)
y = torch.randint(0, 10, (32,))
cac_bo = {
    "SGD": lambda p: torch.optim.SGD(p, lr=0.1),
    "động lượng": lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    "RMSprop": lambda p: torch.optim.RMSprop(p, lr=0.01),
    "Adam": lambda p: torch.optim.Adam(p, lr=1e-3),
    "AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.01),
}
he_so = {}
for ten, tao in cac_bo.items():
    mo_hinh = mlp_784()
    bo = tao(mo_hinh.parameters())
    ham_mat_mat(mo_hinh(x), y).backward()
    bo.step()
    byte_tham_so = sum(p.numel() * p.element_size() for p in mo_hinh.parameters())
    byte_trang_thai, ten_tensor = 0, set()
    for p, tt in bo.state.items():
        for k, v in tt.items():
            if torch.is_tensor(v) and v.shape == p.shape:   # bỏ qua bộ đếm bước, một số vô hướng
                byte_trang_thai += v.numel() * v.element_size()
                ten_tensor.add(k)
    he_so[ten] = (byte_tham_so + byte_trang_thai) / byte_tham_so
    print(f"{ten:<11} trạng thái {byte_trang_thai / 1e6:5.2f} MB  {sorted(ten_tensor)}  →  {he_so[ten]:.0f}× dung lượng tham số")

assert [he_so[t] for t in ("SGD", "động lượng", "RMSprop", "Adam", "AdamW")] == [1, 2, 2, 3, 3]
theo_sach("SGD: tham số + trạng thái", he_so["SGD"], sach=1, nguon="1× · 2× · 2× · 3×")
theo_sach("động lượng", he_so["động lượng"], sach=2, nguon="1× · 2× · 2× · 3×")
theo_sach("RMSprop", he_so["RMSprop"], sach=2, nguon="1× · 2× · 2× · 3×")
theo_sach("Adam", he_so["Adam"], sach=3, nguon="1× · 2× · 2× · 3×")
print(f"(mỗi tensor tham số còn một bộ đếm bước vô hướng trong trạng thái của RMSprop và Adam; không đáng kể)")

Đúng như bảng của chương: động lượng giữ `momentum_buffer`, RMSprop giữ `square_avg`, Adam giữ cả
`exp_avg` (mô-men bậc một) lẫn `exp_avg_sq` (mô-men bậc hai). AdamW tách suy giảm trọng số (weight
decay) ra khỏi bước cập nhật mà không thêm tensor nào, nên vẫn là 3 lần.

Nhân lên. Với một mô hình 100 triệu tham số ở FP32, riêng hai vector mô-men của Adam là bao nhiêu?
Và GPT-2 XL, 1.5 tỷ tham số ở FP32, cần bao nhiêu cho trạng thái tĩnh, tức trọng số, gradient và
trạng thái bộ tối ưu, trước khi đếm giá trị kích hoạt (Napkin Math 1.2)?

In [ ]:
FP32 = sach(4, nguon="4 byte · 2 byte · 10⁹ tham số")
N_100M = sach(100e6, trich="for a 100M-parameter model")
theo_sach("hai vector mô-men, mô hình 100 triệu tham số (MB)", 2 * FP32 * N_100M / 1e6, sach=800, nguon="800 MB")

N_XL = sach(1.5e9, nguon="1,5 tỷ (XL)")
V100_GB = sach(32, nguon="6 + 6 + 12 = 24 GB · 32 GB")
trong_so = N_XL * FP32 / 1e9
theo_sach("trọng số FP32 (GB)", trong_so, sach=6, nguon="6 GB FP32 · 12 GB trạng thái Adam")
theo_sach("trạng thái Adam (GB)", 2 * trong_so, sach=12, nguon="6 GB FP32 · 12 GB trạng thái Adam")
theo_sach("trạng thái tĩnh với Adam (GB)", trong_so * (1 + he_so["Adam"]), sach=24, nguon="6 + 6 + 12 = 24 GB")

ten_bo = ["SGD", "động lượng", "RMSprop", "Adam"]
tinh = {t: trong_so * (1 + he_so[t]) for t in ten_bo}   # trọng số + gradient + trạng thái
fig, ax = plt.subplots(figsize=(6.8, 3.8))
cot = ax.bar(ten_bo, [trong_so] * 4, label="trọng số")
ax.bar(ten_bo, [trong_so] * 4, bottom=[trong_so] * 4, label="gradient")
ax.bar(ten_bo, [tinh[t] - 2 * trong_so for t in ten_bo], bottom=[2 * trong_so] * 4, label="trạng thái bộ tối ưu")
for i, t in enumerate(ten_bo):
    ax.annotate(f"{tinh[t]:.0f} GB", (i, tinh[t]), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
ax.axhline(V100_GB, color="k", ls="--", lw=1)
ax.text(3.45, V100_GB + 0.6, "dung lượng một V100: 32 GB", fontsize=8, ha="right")
ax.set_ylim(0, 36)
ax.set_ylabel("GB, FP32")
ax.set_title("GPT-2 XL, 1.5 tỷ tham số: trạng thái tĩnh theo bộ tối ưu")
ax.legend(loc="upper left", fontsize=8, bbox_to_anchor=(0, 0.86))
plt.tight_layout()
plt.show()

Với Adam, trạng thái tĩnh của GPT-2 XL đã là 24 GB, sát dung lượng 32 GB của một V100, **trước khi**
chỗ lưu giá trị kích hoạt được đếm. Sổ tay 03 đếm nốt phần ấy.

Trạng thái không chỉ chiếm chỗ mà còn phải di chuyển. Theo mô tả của chương, mỗi bước Adam đọc
gradient, tham số, mô-men bậc một và mô-men bậc hai, rồi ghi lại hai mô-men và tham số: 7 tensor cỡ
tham số qua bộ nhớ, so với 3 của SGD (đọc tham số và gradient, ghi tham số). Đo thời gian của riêng
lời gọi `step()` trên một mạng 8.4 triệu tham số:

In [ ]:
lon = nn.Sequential(nn.Linear(2048, 2048), nn.Linear(2048, 2048))
for p in lon.parameters():
    p.grad = torch.randn_like(p)
for ten in ("SGD", "động lượng", "Adam"):
    bo = cac_bo[ten](lon.parameters())
    bo.step()                       # bước đầu tạo trạng thái; không tính
    tot = float("inf")
    for _ in range(5):
        t0 = time.perf_counter()
        bo.step()
        tot = min(tot, time.perf_counter() - t0)
    do_tren_may(f"một lời gọi step() của {ten}", round(tot * 1e3, 2), "ms")

Bước cập nhật làm rất ít phép tính cho mỗi byte nó đọc, nên thời gian của nó đi theo lượng byte
di chuyển hơn là theo số phép tính. Lớp trừu tượng giúp viết gọn, nhưng hai tensor thêm của Adam
vẫn phải chiếm bộ nhớ và đi qua phân cấp bộ nhớ ở mỗi bước.

## 3. Batch: nhiễu của gradient minibatch

Một lần cập nhật minibatch lấy trung bình gradient trên các mẫu trong batch. Với các mẫu độc lập và
phân phối tương tự nhau, độ lệch chuẩn của trung bình ấy giảm xấp xỉ theo $1/\sqrt{B}$. Kiểm trên
dữ liệu thật: tập chữ số viết tay `load_digits` (1,797 ảnh 8×8), một mạng nhỏ cố định, và với mỗi
kích thước batch, 200 minibatch rút ngẫu nhiên (có hoàn lại). Độ lệch được đo bằng chuẩn của hiệu
giữa gradient minibatch và gradient trên toàn tập.

**Dự đoán:** tăng batch lên 4 lần thì độ lệch giảm bao nhiêu lần?

In [ ]:
from sklearn.datasets import load_digits

Xd, yd = load_digits(return_X_y=True)
Xd = torch.tensor(Xd / 16.0, dtype=torch.float32)
yd = torch.tensor(yd)
dat_hat_giong()
nho = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))


def gradient(chi_so):
    nho.zero_grad()
    ham_mat_mat(nho(Xd[chi_so]), yd[chi_so]).backward()
    return torch.cat([p.grad.flatten() for p in nho.parameters()])


g_toan_tap = gradient(torch.arange(len(Xd)))
rut = torch.Generator().manual_seed(SEED)
cac_B = [1, 4, 16, 64, 256]
do_lech = []
for B in cac_B:
    hieu = [(gradient(torch.randint(0, len(Xd), (B,), generator=rut)) - g_toan_tap).norm() ** 2 for _ in range(200)]
    do_lech.append(float(torch.stack(hieu).mean().sqrt()))
    print(f"B = {B:>3}: độ lệch gradient {do_lech[-1]:.4f}")
doc = np.polyfit(np.log(cac_B), np.log(do_lech), 1)[0]
print(f"độ dốc trên thang log-log: {doc:.3f}  (1/√B cho -0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.loglog(cac_B, do_lech, "o-", label="đo trên load_digits, 200 minibatch mỗi điểm")
ax.loglog(cac_B, [do_lech[0] / math.sqrt(B) for B in cac_B], "k--", lw=1, label="1/√B, đặt qua điểm B = 1")
ax.set_xticks(cac_B, [str(B) for B in cac_B])
ax.set_yticks([0.125, 0.25, 0.5, 1, 2], ["0.125", "0.25", "0.5", "1", "2"])
ax.minorticks_off()
ax.set_xlabel("kích thước batch B")
ax.set_ylabel("‖g_B − g‖, căn trung bình bình phương")
ax.set_title("Nhân batch lên 4 lần, nhiễu gradient giảm một nửa")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Nhiễu giảm theo $1/\sqrt{B}$, còn chi phí một bước tăng theo $B$: nhân batch lên 4 lần chỉ làm nhiễu
giảm một nửa. Với một tập dữ liệu cố định, nhân đôi batch còn **giảm một nửa số lần cập nhật** mỗi
epoch: 1,797 ảnh ở batch 32 là 57 bước, ở batch 64 là 29 bước. Chương nhắc rằng thông lượng thường
chỉ tăng theo batch cho tới khi phần cứng được lấp đủ (sổ tay 01 đo đúng đường cong ấy), còn số lần
và chất lượng cập nhật cần để hội tụ thì đi theo một đường cong riêng của từng tải công việc.

## 4. Lịch tốc độ học

Cấu hình GPT-2 XL của chương cho tốc độ học khởi động dần (warmup) từ 0 lên $2.5\cdot 10^{-4}$ trong
500 bước đầu, rồi giảm dần theo cosin (cosine annealing). Chương không nói tổng số bước của cấu hình
ấy; ô dưới **giả định** 50,000 bước, bằng kịch bản minh hoạ của Napkin Math 1.1, để vẽ cả đường.
Lịch được dựng bằng `torch.optim.lr_scheduler.LambdaLR`, và mỗi giá trị vẽ ra là tốc độ học mà bộ
tối ưu thật sự nhận ở bước đó. Hai lịch khác mà chương kể tên, suy giảm theo hàm mũ và giảm theo bậc,
vẽ cùng đỉnh để so.

In [ ]:
LR_DINH = sach(2.5e-4, nguon="β₁=0,9 · β₂=0,999 · 2,5e-4 · 500 bước")
KHOI_DONG = sach(500, nguon="2,5e-4 · 500 bước")
TONG_BUOC = sach(50_000, nguon="16041,7 PFLOP · 50.000 bước")   # giả định: chương không nói tổng số bước


def he_so_lich(buoc):
    if buoc < KHOI_DONG:
        return buoc / KHOI_DONG
    tien_do = (buoc - KHOI_DONG) / (TONG_BUOC - KHOI_DONG)
    return 0.5 * (1 + math.cos(math.pi * min(tien_do, 1.0)))


def chay_lich(tao_lich, so_buoc, lich_bat_dau=None, bo_bat_dau=None):
    """Tốc độ học mà bộ tối ưu nhận ở từng bước. Không cần mô hình: một tham số giả là đủ."""
    w = torch.zeros(1, requires_grad=True)
    bo = torch.optim.AdamW([w], lr=LR_DINH, betas=(BETA1, BETA2))
    lich = tao_lich(bo)
    if bo_bat_dau is not None:
        bo.load_state_dict(bo_bat_dau)
    if lich_bat_dau is not None:
        lich.load_state_dict(lich_bat_dau)
    gia_tri = []
    for _ in range(so_buoc):
        gia_tri.append(bo.param_groups[0]["lr"])
        bo.step()
        lich.step()
    return gia_tri, bo, lich


cosin, _, _ = chay_lich(lambda bo: torch.optim.lr_scheduler.LambdaLR(bo, he_so_lich), TONG_BUOC)
mu, _, _ = chay_lich(lambda bo: torch.optim.lr_scheduler.ExponentialLR(bo, gamma=0.99995), TONG_BUOC)
bac, _, _ = chay_lich(lambda bo: torch.optim.lr_scheduler.StepLR(bo, step_size=15_000, gamma=0.3), TONG_BUOC)
assert cosin[0] == 0.0 and abs(cosin[KHOI_DONG] - LR_DINH) < 1e-15 and cosin[-1] < 1e-9

from matplotlib.ticker import FuncFormatter

khoa_hoc = FuncFormatter(lambda v, _: f"{v:.1e}" if v else "0")       # 2.5e-04 thay cho 0.00025
nghin = FuncFormatter(lambda v, _: f"{v:,.0f}")                      # 50,000
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
for a in (ax, ax2):
    a.yaxis.set_major_formatter(khoa_hoc)
    a.xaxis.set_major_formatter(nghin)
ax.plot(cosin, label="khởi động dần 500 bước + cosin (GPT-2 XL)")
ax.plot(mu, label="suy giảm theo hàm mũ, γ = 0.99995")
ax.plot(bac, label="giảm theo bậc, ×0.3 mỗi 15,000 bước")
ax.set_xlabel("bước")
ax.set_ylabel("tốc độ học")
ax.set_title("Ba lịch, cùng đỉnh 2.5e-4")
ax.legend(loc="upper right", fontsize=8)
ax2.plot(range(1500), cosin[:1500])
ax2.axvline(KHOI_DONG, color="0.5", ls="--", lw=1)
ax2.annotate("đỉnh 2.5e-4 ở bước 500", (KHOI_DONG, LR_DINH), xytext=(30, -30), textcoords="offset points",
             fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5"))
ax2.set_xlabel("bước")
ax2.set_title("1,500 bước đầu: pha khởi động dần")
plt.tight_layout()
plt.show()

### Lịch là trạng thái, nên phải vào checkpoint

Bộ lập lịch không thêm trạng thái cho mỗi tham số, nhưng nó theo dõi bước hiện tại, và trạng thái ấy
phải được phục hồi cùng với checkpoint. Ô dưới dừng ở bước 1,000, lưu `state_dict()` của bộ tối ưu
và bộ lập lịch, dựng lại cả hai như sau một lần khởi động lại, rồi chạy tiếp. Lần thứ hai quên phục
hồi bộ lập lịch.

In [ ]:
TAO_LICH = lambda bo: torch.optim.lr_scheduler.LambdaLR(bo, he_so_lich)
dung = 1000
_, bo_cu, lich_cu = chay_lich(TAO_LICH, dung)
tiep_dung, _, _ = chay_lich(TAO_LICH, 2000, lich_bat_dau=lich_cu.state_dict(), bo_bat_dau=bo_cu.state_dict())
tiep_quen, _, _ = chay_lich(TAO_LICH, 2000, bo_bat_dau=bo_cu.state_dict())
assert tiep_dung == cosin[dung:dung + 2000]     # khớp từng bước với lần chạy không bị ngắt
print(f"phục hồi đủ:      tốc độ học ở bước 1,001 là {tiep_dung[1]:.3e}, đúng như lần chạy không bị ngắt")
print(f"quên bộ lập lịch: tốc độ học ở bước 1,001 là {tiep_quen[1]:.3e}, tức khởi động dần lại từ đầu")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(3000), cosin[:3000], color="0.7", lw=4, label="chạy liền một mạch")
ax.plot(range(dung, dung + 2000), tiep_dung, color="C0", lw=1.2, label="khởi động lại, phục hồi cả bộ lập lịch")
ax.plot(range(dung, dung + 2000), tiep_quen, color="C3", lw=1.2, ls="--", label="khởi động lại, quên bộ lập lịch")
ax.axvline(dung, color="0.5", lw=1, ls=":")
ax.text(dung + 30, LR_DINH * 1.04, "checkpoint ở bước 1,000", fontsize=8)
ax.yaxis.set_major_formatter(khoa_hoc)
ax.xaxis.set_major_formatter(nghin)
ax.set_xlabel("bước")
ax.set_ylabel("tốc độ học")
ax.set_ylim(0, LR_DINH * 1.12)
ax.set_title("Bộ lập lịch là trạng thái của lần chạy")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### Vì sao không đặt tốc độ học một lần rồi thôi

Một ngộ nhận thường gặp cho rằng chỉ cần đặt tốc độ học một lần rồi không bao giờ đổi. Chương giải
thích rằng một tốc độ học tĩnh vừa đúng ở những bước đầu sẽ quá lớn ở giai đoạn sau. Thấy được điều
đó cần một bài toán có nhiễu thật: hồi quy tuyến tính 20 chiều trên 4,000 mẫu có nhiễu, huấn luyện
bằng SGD với minibatch 8. Thước đo là khoảng cách tới nghiệm bình phương tối thiểu, điểm tốt nhất
mà dữ liệu này cho phép.

**Dự đoán:** với tốc độ học cố định 0.05, khoảng cách ấy có về 0 không nếu chạy đủ lâu?

In [ ]:
sinh = torch.Generator().manual_seed(SEED)
Xr = torch.randn(4000, 20, generator=sinh)
yr = Xr @ torch.randn(20, generator=sinh) + 0.5 * torch.randn(4000, generator=sinh)
w_tot = torch.linalg.lstsq(Xr, yr.unsqueeze(1)).solution.squeeze()
SO_BUOC = 1500


def hoi_quy(lr, lich=None):
    w = torch.zeros(20, requires_grad=True)
    bo = torch.optim.SGD([w], lr=lr)
    lap_lich = torch.optim.lr_scheduler.LambdaLR(bo, lich) if lich else None
    rut = torch.Generator().manual_seed(SEED)
    cach = []
    for _ in range(SO_BUOC):
        i = torch.randint(0, 4000, (8,), generator=rut)
        bo.zero_grad()
        (0.5 * ((Xr[i] @ w - yr[i]) ** 2).mean()).backward()
        bo.step()
        if lap_lich:
            lap_lich.step()
        cach.append(float((w.detach() - w_tot).norm()))
    return cach


cosin_100 = lambda b: b / 100 if b < 100 else 0.5 * (1 + math.cos(math.pi * (b - 100) / (SO_BUOC - 100)))
lan_chay = {
    "cố định 0.1": hoi_quy(0.1),
    "cố định 0.05": hoi_quy(0.05),
    "cố định 0.005": hoi_quy(0.005),
    "đỉnh 0.05, khởi động dần 100 bước + cosin": hoi_quy(0.05, cosin_100),
}
fig, ax = plt.subplots(figsize=(7, 4))
for ten, c in lan_chay.items():
    ax.semilogy(c, lw=1.2, label=f"{ten}: cuối {c[-1]:.3f}")
ax.set_xlabel("bước SGD, minibatch 8")
ax.set_ylabel("‖w − w*‖ (thang log)")
ax.set_title("Tốc độ học cố định dừng ở một sàn nhiễu")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Tốc độ học lớn cố định đi xuống rất nhanh rồi dừng lại ở một **sàn**: mỗi bước nhiễu đẩy $w$ ra khỏi
nghiệm một đoạn tỉ lệ với tốc độ học, và sàn càng cao khi tốc độ học càng lớn. Tốc độ học nhỏ cố
định thì xuống thấp hơn nhưng rất chậm. Lịch lấy được cả hai: đầu lần chạy nó xuống nhanh như tốc độ
học lớn, cuối lần chạy nó về gần sàn của tốc độ học nhỏ.

Một ngộ nhận liên quan cho rằng siêu tham số chuyển thẳng được từ thí nghiệm nhỏ sang lần huấn luyện
lớn. Khi batch tăng tám lần, từ 512 lên 4,096, quy tắc co giãn tuyến tính (linear scaling rule)
nhân tốc độ học theo đúng tỉ lệ đó:

In [ ]:
B_NHO = sach(512, nguon="512 → 4.096 · 0,1 → 0,8")
B_LON = sach(4096, nguon="512 → 4.096 · 0,1 → 0,8")
LR_NHO = sach(0.1, nguon="512 → 4.096 · 0,1 → 0,8")
theo_sach("tốc độ học ở batch 4,096", LR_NHO * B_LON / B_NHO, sach=0.8, nguon="512 → 4.096 · 0,1 → 0,8")
print(f"số lần cập nhật mỗi epoch giảm {B_LON // B_NHO} lần, nên mỗi lần cập nhật phải đi xa hơn")

Chương nói rõ đó là một recipe thực nghiệm, thường cần khởi động dần, không phải một bảo đảm hội tụ.
Hình hồi quy ở trên cũng cho thấy vì sao: tốc độ học 0.1 ở đó đã tạo sàn nhiễu cao hơn 0.05.

## 5. Tích luỹ gradient

Khi batch hiệu dụng không vừa bộ nhớ, hệ thống chia nó thành $k$ micro-batch cỡ $b$ chạy tuần tự,
cộng dồn gradient vào bộ đệm `.grad` mà không cập nhật, rồi mới gọi `step()`. Phương trình 17 của
chương nói gradient tích luỹ đúng bằng gradient của cả batch:

$$\nabla \mathcal{L}_B = \frac{1}{B}\sum_{i=1}^{B} \nabla \mathcal{L}_i = \frac{1}{k}\sum_{j=1}^{k}\left(\frac{1}{b}\sum_{i \in \mathcal{B}_j} \nabla \mathcal{L}_i\right)$$

Đó cũng là lý do `zero_grad()` tồn tại: gradient cộng dồn qua các lời gọi lượt ngược, và chính hành
vi ấy được dùng có chủ ý ở đây. Ô dưới so ba cách tính gradient cho cùng 64 mẫu: một batch 64; bốn
micro-batch 16 với mất mát chia cho $k$; và bốn micro-batch quên chia.

**Dự đoán:** cách thứ hai có cho **đúng từng bit** gradient của cách thứ nhất không?

In [ ]:
def mang(chuan_hoa_batch=False):
    dat_hat_giong()
    if chuan_hoa_batch:
        return nn.Sequential(nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128, 10))
    return nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))


def gom(m):
    return torch.cat([p.grad.flatten() for p in m.parameters()])


B_HD, k = 64, 4
b = B_HD // k
lo = torch.arange(B_HD)
for bn in (False, True):
    m = mang(bn)
    m.zero_grad()
    ham_mat_mat(m(Xd[lo]), yd[lo]).backward()
    g_lon = gom(m).clone()

    m.zero_grad()
    for j in range(k):
        phan = lo[j * b:(j + 1) * b]
        (ham_mat_mat(m(Xd[phan]), yd[phan]) / k).backward()   # chia cho k: trung bình của k trung bình
    g_tich_luy = gom(m).clone()

    m.zero_grad()
    for j in range(k):
        phan = lo[j * b:(j + 1) * b]
        ham_mat_mat(m(Xd[phan]), yd[phan]).backward()          # quên chia cho k
    g_quen = gom(m).clone()

    lech = float((g_tich_luy - g_lon).norm() / g_lon.norm())
    print(f"{'có' if bn else 'không có'} chuẩn hoá theo batch:")
    print(f"   tích luỹ đúng cách: lệch tương đối {lech:.1e}, trùng từng bit: {torch.equal(g_tich_luy, g_lon)}")
    print(f"   quên chia cho k:    chuẩn gradient gấp {float(g_quen.norm() / g_lon.norm()):.2f} lần")
    if not bn:
        assert lech < 1e-5 and abs(float(g_quen.norm() / g_lon.norm()) - k) < 1e-4

Ba bài học, đúng ba điều kiện mà chương đặt ra:

* Không có tầng có trạng thái, tích luỹ cho cùng gradient **tới sai số làm tròn**, không phải tới
  từng bit: thứ tự cộng đã đổi. Chương yêu cầu hội tụ trong một sai số đã khai báo, không đòi trùng bit.
* Quên chia cho $k$ thì gradient lớn gấp đúng $k$ lần, tức tốc độ học hiệu dụng bị nhân $k$ mà không
  ai đổi nó. Chương gọi việc chuẩn hoá mất mát là nguồn lỗi tinh vi thường gặp.
* Chuẩn hoá theo batch (batch normalization) tính thống kê trên **micro-batch**, không phải trên cả
  batch hiệu dụng, nên gradient khác hẳn. Dropout và tăng cường dữ liệu ngẫu nhiên cũng phá đẳng thức.

### Bộ nhớ và thời gian

Tích luỹ đổi bộ nhớ lấy thời gian. Bộ nhớ giá trị kích hoạt đi theo $b$ chứ không theo $B$: ô dưới
đếm byte của mọi tensor mà autograd **giữ lại cho lượt ngược**, bằng `saved_tensors_hooks`, rồi
đo thời gian một lần cập nhật với batch hiệu dụng 256 chia thành micro-batch cỡ khác nhau.

**Dự đoán:** với micro-batch 1 (256 lượt xuôi ngược trước mỗi `step()`), một lần cập nhật chậm hơn
micro-batch 256 bao nhiêu lần?

In [ ]:
mlp_lon = nn.Sequential(nn.Linear(64, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
bo = torch.optim.SGD(mlp_lon.parameters(), lr=0.01)
la_tham_so = {p.data_ptr() for p in mlp_lon.parameters()}


def byte_giu_lai(mo_hinh, x):
    """Byte của các tensor autograd giữ lại cho lượt ngược (không tính tham số), mỗi tensor một lần."""
    giu = {}

    def goi(t):
        if t.data_ptr() not in la_tham_so:
            giu[t.data_ptr()] = max(giu.get(t.data_ptr(), 0), t.numel() * t.element_size())
        return t

    with torch.autograd.graph.saved_tensors_hooks(goi, lambda t: t):
        mo_hinh(x)
    return sum(giu.values())


E = 256
Xe, ye = Xd[torch.arange(E)], yd[torch.arange(E)]
cac_b = [1, 4, 16, 64, 256]
kich_hoat, ms_cap_nhat = [], []
for bb in cac_b:
    kk = E // bb
    kich_hoat.append(byte_giu_lai(mlp_lon, Xe[torch.arange(bb)]))

    def cap_nhat():
        bo.zero_grad()
        for j in range(kk):
            (ham_mat_mat(mlp_lon(Xe[j * bb:(j + 1) * bb]), ye[j * bb:(j + 1) * bb]) / kk).backward()
        bo.step()

    cap_nhat()
    tot = float("inf")
    for _ in range(3):
        t0 = time.perf_counter()
        cap_nhat()
        tot = min(tot, time.perf_counter() - t0)
    ms_cap_nhat.append(tot * 1e3)
    print(f"micro-batch {bb:>3} × {kk:>3} lần: giữ lại {kich_hoat[-1] / 1024:7.1f} KiB giá trị kích hoạt")
    do_tren_may(f"micro-batch {bb}, một lần cập nhật", round(tot * 1e3, 1), "ms")

# Bộ nhớ giá trị kích hoạt tỉ lệ đúng với micro-batch: b nhân 4 thì số byte nhân 4.
assert all(kich_hoat[i + 1] == 4 * kich_hoat[i] for i in range(len(cac_b) - 1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.loglog(cac_b, [kb / kich_hoat[-1] for kb in kich_hoat], "o-", label="giá trị kích hoạt giữ lại (đếm)")
ax.loglog(cac_b, [t / ms_cap_nhat[-1] for t in ms_cap_nhat], "s-", label="thời gian một lần cập nhật (đo trên máy này)")
ax.set_xticks(cac_b, [str(bb) for bb in cac_b])
ax.set_xlabel("kích thước micro-batch b (batch hiệu dụng luôn là 256)")
ax.set_ylabel("so với b = 256")
ax.set_title("Micro-batch nhỏ: ít bộ nhớ hơn, chậm hơn")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Tổng số FLOP không đổi, vì cả 256 mẫu vẫn được xử lý. Nhưng micro-batch nhỏ dùng bộ tăng tốc
(accelerator) kém hơn và gọi kernel nhiều lần hơn, nên thời gian mỗi lần cập nhật tăng. Chương nói mức
phạt ấy phải đo, không suy ra được từ hệ số giảm bộ nhớ. Ở một bản phân tích minh hoạ của chương,
batch hiệu dụng 512 với batch vật lý 64 đổi 5 phần trăm hiệu suất lấy mức giảm bộ nhớ 8 lần:

In [ ]:
theo_sach("mức giảm bộ nhớ giá trị kích hoạt", sach(512, nguon="512 · 64 · 5 phần trăm") / sach(64, nguon="512 · 64 · 5 phần trăm"),
          sach=8, nguon="512 · 64 · 5 phần trăm · 8 lần")
print("(mức phạt 5 phần trăm là số của bản phân tích ấy; ở trên máy này, hãy đọc nó từ hình vừa vẽ)")

### Kế hoạch tích luỹ cho GPT-2 (Napkin Math 1.8)

Kịch bản minh hoạ của chương, với giá và thời lượng giả định: công thức bộ nhớ đơn giản hoá của chương
cho một V100 32 GB giữ được micro-batch 4 mỗi GPU, và mục tiêu là batch hiệu dụng 512.

In [ ]:
MICRO = sach(4, nguon="4 · 16 → 64 mỗi GPU · 8 GPU → 512")
K_TICH_LUY = sach(16, nguon="4 · 16 → 64 mỗi GPU · 8 GPU → 512")
N_GPU = sach(8, nguon="4 · 16 → 64 mỗi GPU · 8 GPU → 512")
B_HIEU_DUNG = sach(512, nguon="512 ÷ 4 = 128 GPU")
GIA_GIO = sach(16, nguon="16 đô la mỗi giờ")
CHI_PHI = sach(688_000, nguon="688 nghìn đô la")

gpu_thuong_tru = B_HIEU_DUNG // MICRO
theo_sach("GPU nếu giữ cả 512 mẫu cùng lúc", gpu_thuong_tru, sach=128, nguon="512 ÷ 4 = 128 GPU")
theo_sach("mẫu mỗi GPU mỗi lần cập nhật", MICRO * K_TICH_LUY, sach=64, nguon="4 · 16 → 64 mỗi GPU")
theo_sach("batch toàn cục", N_GPU * MICRO * K_TICH_LUY, sach=512, nguon="8 GPU → 512")
gio_thuong_tru, gio_tich_luy = gpu_thuong_tru * GIA_GIO, N_GPU * GIA_GIO
theo_sach("cấu hình thường trú (đô la mỗi giờ)", gio_thuong_tru, sach=2048, nguon="2.048 so với 128 đô la mỗi giờ")
theo_sach("cấu hình tích luỹ (đô la mỗi giờ)", gio_tich_luy, sach=128, nguon="2.048 so với 128 đô la mỗi giờ")
theo_sach("tiết kiệm mỗi giờ (đô la)", gio_thuong_tru - gio_tich_luy, sach=1920, nguon="128 đô la mỗi giờ · 1.920")
theo_sach("tiết kiệm mỗi giờ (%)", 100 * (1 - gio_tich_luy / gio_thuong_tru), sach=93.8, nguon="1.920 · 93,8 phần trăm")

# Cùng lượng công việc: 16 lần ít GPU hơn thì lâu hơn 16 lần, nên tổng tiền như nhau.
h = CHI_PHI / gio_thuong_tru
print(f"thường trú: {h:,.0f} giờ × {gio_thuong_tru:,} đô la = {h * gio_thuong_tru:,.0f} đô la")
print(f"tích luỹ:   {h * gpu_thuong_tru / N_GPU:,.0f} giờ × {gio_tich_luy:,} đô la = {h * gpu_thuong_tru / N_GPU * gio_tich_luy:,.0f} đô la")

Tích luỹ giảm số GPU và tiền đốt mỗi giờ, nhưng co giãn lý tưởng giữ nguyên số giờ-GPU: cùng 688
nghìn đô la, chỉ kéo dài gấp 16 lần. Chi phí thật còn tuỳ mức sử dụng của micro-batch nhỏ (đúng thứ
bạn vừa đo) và truyền thông.

## Thử thêm

1. Ở phần 1, đổi hệ số 20 trong `f_do_choi` (và trong ô vẽ) thành 2. Khi hai chiều có thang gần nhau,
   khoảng cách giữa SGD và RMSprop còn lớn không?
2. Ở phần 3, rút minibatch **không hoàn lại** với `torch.randperm(len(Xd))[:B]`. Ở $B = 256$, tức một
   phần bảy tập dữ liệu, độ lệch nhỏ hơn đường $1/\sqrt{B}$ một chút. Vì sao?
3. Ở phần 5, thay `nn.BatchNorm1d(128)` bằng `nn.LayerNorm(128)`. Chuẩn hoá theo tầng tính thống kê
   trên từng mẫu, vậy tích luỹ có trở lại đúng không?

## Tóm lại

* Tham số cộng trạng thái là 1 lần dung lượng tham số với SGD, 2 lần với động lượng và RMSprop, 3 lần
  với Adam. Với GPT-2 XL ở FP32, trạng thái tĩnh dùng Adam là 24 GB, trước khi đếm giá trị kích hoạt.
* Nhiễu gradient giảm theo $1/\sqrt{B}$, còn số lần cập nhật mỗi epoch giảm theo $B$.
* Lịch tốc độ học là trạng thái: quên phục hồi nó sau một lần khởi động lại thì lần chạy khởi động dần
  lại từ 0. Tốc độ học cố định dừng ở một sàn nhiễu mà lịch vượt qua được.
* Tích luỹ gradient cho cùng gradient tới sai số làm tròn khi mất mát được chia cho $k$ và không có
  tầng có trạng thái như chuẩn hoá theo batch. Nó giảm bộ nhớ giá trị kích hoạt $k$ lần và làm mỗi lần
  cập nhật chậm hơn.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Huấn luyện mô hình — khi định luật sắt trở thành công cụ hằng ngày](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch08-model-training/), dựng từ chương
[*Model Training*](https://mlsysbook.ai/vol1/training/training.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_08_model_train.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.